In [0]:
--dim country
CREATE TABLE IF NOT EXISTS
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_country
USING DELTA
AS
SELECT
    country_code,
    AREA AS country_name,
    region,
    subregion
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_silver.scd1
WHERE AREA <> 'Russian Federation';

SELECT *
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_country
ORDER BY country_name;

--dim_crop
CREATE TABLE IF NOT EXISTS
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_crop
USING DELTA
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY ITEM) AS crop_key,
    ITEM AS crop_name
FROM (
    SELECT DISTINCT TRIM(ITEM) AS ITEM
    FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_silver.yield_and_climate
    WHERE ITEM IS NOT NULL
      AND TRIM(ITEM) <> ''
);

SELECT *
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_crop
ORDER BY crop_key;

--dim_year
CREATE TABLE IF NOT EXISTS
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_year
USING DELTA
AS
SELECT DISTINCT
    CAST(YEAR AS INT) AS year_key,
    CAST(YEAR AS INT) AS year
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_silver.yield_and_climate
WHERE YEAR IS NOT NULL;

SELECT *
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_year
ORDER BY year;

--fact_table
CREATE TABLE IF NOT EXISTS
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.fact_agri_environment
USING DELTA
AS
SELECT
    c.country_code AS country_key,
    cr.crop_key,
    y.year_key,

    a.YIELD AS crop_yield,
    a.AREA_HARVESTED AS area_harvested,
    a.PRODUCTION_QUANTITY AS production_quantity,

    e.freshwater_per_capita,
    e.freshwater_withdrawals,
    e.agricultural_land,
    e.forest_area

FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_silver.yield_and_climate a

INNER JOIN dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_country c
    ON TRIM(a.AREA) = c.country_name

INNER JOIN dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_crop cr
    ON TRIM(a.ITEM) = cr.crop_name

INNER JOIN dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_year y
    ON a.YEAR = y.year_key

LEFT JOIN dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_silver.environment e
    ON TRIM(a.AREA) = TRIM(e.AREA)
   AND a.YEAR = e.YEAR;


SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT country_key) AS countries,
    COUNT(DISTINCT crop_key) AS crops,
    COUNT(DISTINCT year_key) AS years,
    COUNT_IF(forest_area IS NULL) AS missing_environment_rows
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.fact_agri_environment;


SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (country_key, crop_key, year_key)) AS unique_business_keys,
    COUNT_IF(
        country_key IS NULL
        OR crop_key IS NULL
        OR year_key IS NULL
    ) AS missing_keys
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.fact_agri_environment;

-- aggregated gold table
CREATE TABLE IF NOT EXISTS
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.agg_crop_year
USING DELTA
AS
SELECT
    f.year_key AS year,
    c.crop_name,

    COUNT(DISTINCT f.country_key) AS country_count,
    SUM(f.production_quantity) AS total_production,
    SUM(f.area_harvested) AS total_area_harvested,

    CASE
        WHEN SUM(f.area_harvested) > 0
        THEN SUM(f.production_quantity) / SUM(f.area_harvested)
        ELSE NULL
    END AS production_per_harvested_area,

    AVG(f.crop_yield) AS avg_reported_yield

FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.fact_agri_environment f

JOIN dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_crop c
    ON f.crop_key = c.crop_key

GROUP BY
    f.year_key,
    c.crop_name;

SELECT
    year,
    crop_name,
    country_count,
    total_production,
    total_area_harvested,
    production_per_harvested_area,
    avg_reported_yield
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.agg_crop_year
ORDER BY year, crop_name;

-- agg_country_year
CREATE TABLE IF NOT EXISTS
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.agg_country_year
USING DELTA
AS
SELECT
    f.country_key,
    c.country_name,
    c.region,
    c.subregion,
    f.year_key AS year,

    COUNT(*) AS crop_count,

    SUM(f.production_quantity) AS total_production,
    SUM(f.area_harvested) AS total_area_harvested,

    CASE
        WHEN SUM(f.area_harvested) > 0
        THEN SUM(f.production_quantity) / SUM(f.area_harvested)
        ELSE NULL
    END AS production_per_harvested_area,

    MAX(f.freshwater_per_capita) AS freshwater_per_capita,
    MAX(f.freshwater_withdrawals) AS freshwater_withdrawals,
    MAX(f.agricultural_land) AS agricultural_land,
    MAX(f.forest_area) AS forest_area

FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.fact_agri_environment f

JOIN dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_country c
    ON f.country_key = c.country_code

GROUP BY
    f.country_key, c.country_name, c.region, c.subregion, f.year_key;

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (country_key, year)) AS unique_country_year,
    COUNT_IF(forest_area IS NULL) AS missing_forest_area
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.agg_country_year;


SELECT
    'dim_country' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT country_code) AS unique_keys,
    COUNT_IF(country_code IS NULL) AS null_keys
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_country

UNION ALL

SELECT
    'dim_crop',
    COUNT(*),
    COUNT(DISTINCT crop_key),
    COUNT_IF(crop_key IS NULL)
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.dim_crop

UNION ALL



